In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from google.adk.agents import LlmAgent

In [8]:
from google import genai

# Initialize the client. It automatically picks up the GEMINI_API_KEY environment variable.
client = genai.Client()

def llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    return response.text
    # return response

In [9]:
question = "I discovered a new llmops course. can I join the course"
answer = llm(question)
print(answer)

That's fantastic! LLMOps is a rapidly growing and incredibly valuable field.

Whether you **can** join the course depends entirely on the specific course you've found and its enrollment requirements.

To figure this out, you'll need to look for information on the official course page or platform where you discovered it. Here's what you should typically look for:

1.  **Enrollment Status:** Is enrollment currently open? Some courses have specific start dates or application windows.
2.  **Prerequisites:** Does the course require specific prior knowledge or skills? This could include:
    *   Python programming
    *   Machine Learning (ML) fundamentals
    *   DevOps principles
    *   Cloud computing experience (AWS, Azure, GCP)
    *   Basic understanding of Large Language Models (LLMs)
3.  **Cost:** Is it a free course, a paid course, or part of a subscription service?
4.  **Application Process:** Do you simply enroll directly, or is there an application process (e.g., submitting a re

### Above results is without context. LLM doesn't know the context so response is very vauge.

## 1.3 RAG - provide hard coded context text as an example

In [10]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [11]:
# question = "I discovered a new llmops course. can I join the course"
question = "I found new LLMops course. How to get the certificate."

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question: {question}

Context: {context}
"""




In [12]:
answer = llm(prompt)
print(answer)

To get a certificate, you need to submit your project while submissions are still being accepted.


## 1.4 RAG - Add search Library 

In [13]:
import requests, json

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

#print(courses_raw)
print(len(courses_raw))
print(json.dumps(courses_raw, indent=2))

4
[
  {
    "course": "machine-learning-zoomcamp",
    "course_name": "ML Zoomcamp",
    "path": "/json/machine-learning-zoomcamp.json",
    "questions_count": 472
  },
  {
    "course": "llm-zoomcamp",
    "course_name": "LLM Zoomcamp",
    "path": "/json/llm-zoomcamp.json",
    "questions_count": 79
  },
  {
    "course": "data-engineering-zoomcamp",
    "course_name": "Data Engineering Zoomcamp",
    "path": "/json/data-engineering-zoomcamp.json",
    "questions_count": 402
  },
  {
    "course": "mlops-zoomcamp",
    "course_name": "MLOps Zoomcamp",
    "path": "/json/mlops-zoomcamp.json",
    "questions_count": 255
  }
]


In [14]:
import json

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""    # example: https://datatalks.club/faq/json/llm-zoomcamp.json

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)   # exapnd the passed list and merge to the source list

len(documents)
# documents[1]
print(json.dumps(documents[1], indent=2))

{
  "id": "226a4baf2f",
  "course": "machine-learning-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "What\u2019s new in the 2025 edition?",
  "answer": "- Deployment module updated to **FastAPI** (replacing Flask) and new tools.\n- Neural networks taught with **PyTorch** (theory videos in Keras are kept; an additional PyTorch implementation video is provided).\n- Deep learning deployment uses **ONNX Runtime** on AWS Lambda (replacing TensorFlow Lite)."
}


In [15]:
# 1.5 Add search Library

import sys
import site

print(sys.executable)   # Check the executable package
site.main()             # This forces Python to re-een its site-packages directoriest


from minsearch import Index
index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)
# select * from index where course = "machine-learning-zoomcamp"


index.fit(documents)



/home/ashok/Ashok/work/my-llm-zoomcamp-2026/.venv/bin/python


In [16]:
# question = "What\u2019s new in the 2025 edition?"
question = "How to get the course completion certificate"
# index.search(sample_question)


def search(question):
    boost_dict={'question': 2.0, 'section' : 0.4 }       # 1 is 100% standard weight, 2 mean 200% weight
    filter_dict={'course' : 'llm-zoomcamp'}                # Limiting the search to LLM Zoom Camp course only

    return index.search(
        question, 
        boost_dict = boost_dict,
        filter_dict = filter_dict,
        num_results = 5)
    
search_results = search(question)
print (json.dumps(search_results, indent=2))


[
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "9f689c185f",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I missed the first homework - can I still get a certificate?",
    "answer": "Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderb

In [17]:
## 1.7 Building Prompt

In [18]:
# LLM its role and how to answer

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

# Placeholders for the question and the context:

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""


# Pre-processing step - Convert List of dict (output of search) into a string

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()


In [19]:

# Building the prompt - by combining question with the search results

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()


#question = "How to get the course completion certificate"
#question = "How is the weather in newyork city"
question = "When does the next course starts"
search_results= search(question)

prompt = build_prompt(question, search_results)
print(prompt)

Question:
When does the next course starts

Context:
General Course-Related Questions
Q: When will the course be offered next?
A: Summer 2025.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Server Error (500) When logging in to course homework using GitHub
A: Additional error text seen:

```
Third-Party Login Failure

An error occurred while attempting to login via your third-party account.
```

The current solution is to use Google or Slack to log in and submit homework answers, as the root cause analysis for the GitHub issue is sporadic and doesn’t impact all users.

General Course-Related Questions
Q: Certifica

In [25]:
from google.adk.agents import LlmAgent
from google.genai import types
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

# Dummy placeholders for your local functions (keep your actual ones)
# INSTRUCTIONS = "You are a helpful FAQ assistant."
# def search(q): return "Next course starts Summer 2026."
# def build_prompt(q, r): return f"Context: {r}\nQuestion: {q}"


APP_NAME = "research_team_app"
USER_ID  = "user_001"

question = "When does the next course starts"
question = "Do I get certification after completion"
question = "Are there any lectures/videos? Where are they?"
question = "can I use google ADK for this course"       # not in FAQ
question = "can I use Groq or ollama for this course"
#question = "what courses are offered"

search_results= search(question)
prompt = build_prompt(question, search_results)

# 1. Define the Agent Architecture using LlmAgent
support_agent = LlmAgent(
    name="faq_agent",  # <--- Note this name
    model="gemini-2.5-flash-lite",
    instruction=INSTRUCTIONS
)

session_service = InMemorySessionService()
runner = Runner(
    agent=support_agent,            # Agent selection here...
    app_name=APP_NAME,
    session_service=session_service
)
await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id="session_001"
)

async for event in runner.run_async(
    user_id=USER_ID,
    session_id="session_001",
    new_message=Content(parts=[Part(text=prompt)])
):
    # Print which agent is currently active
    if hasattr(event, "author") and event.author:
        print(f"  [{event.author}] working...")
    if event.is_final_response():
        print("\nFINAL REPORT\n" + "=" * 40)
        print(event.content.parts[0].text)
# Inspect what ended up in state after the full run
final = await session_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id="session_001"
)
print(f"\nState snapshot: {list(final.state.keys())}")

  [faq_agent] working...

FINAL REPORT
Yes, you can use Groq or Ollama for this course. The homework is designed to be completed locally without paid services, although you may need to adjust the code. Some Groq models also offer free tool use support for development.

State snapshot: []
